# OF Cleanup Regression Smoke Test

Compares **baseline** `reusable/OptimumFilter.py` (before cleanup) vs **cleaned** `reusable/OF_tutorial/OptimumFilter.py` for:
- Numerical equivalence on shared-valid paths (`fit`, `fit_with_shift`, `sliding_fit` with `chisq_mode='all'`, `chisq_at_indices`)
- Runtime comparison after JIT warm-up

`chisq_mode='none'` is reported separately because it was intentionally fixed in the cleaned version.

In [1]:
from __future__ import annotations

import importlib.util
import statistics
import time
from pathlib import Path

import numpy as np

BASE_DIR = Path.cwd()
REPO_ROOT = BASE_DIR.parent
BASELINE_PATH = REPO_ROOT / 'OptimumFilter.py'
CLEAN_PATH = BASE_DIR / 'OptimumFilter.py'

print('baseline:', BASELINE_PATH)
print('cleaned :', CLEAN_PATH)
assert BASELINE_PATH.exists(), BASELINE_PATH
assert CLEAN_PATH.exists(), CLEAN_PATH

baseline: /home/dwong/DELight_mtr/PCA_dev/reusable/OptimumFilter.py
cleaned : /home/dwong/DELight_mtr/PCA_dev/reusable/OF_tutorial/OptimumFilter.py


In [2]:
def load_module(module_path: Path, module_name: str):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Cannot load module from {module_path}')
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def maybe_load_reference_inputs():
    template_path = Path('/home/dwong/DELight_mtr/trigger_study/archive/wk15/templates/vac_ch_template.npy')
    noise_psd_path = Path('/home/dwong/DELight_mtr/trigger_study/archive/wk29/pink_psd.npy')
    fs = 250000.0

    if template_path.exists() and noise_psd_path.exists():
        template = np.load(template_path).astype(np.float64)
        noise_psd = np.load(noise_psd_path).astype(np.float64)
        src = 'reference files'
    else:
        rng = np.random.default_rng(42)
        n = 4096
        x = np.arange(n, dtype=np.float64)
        template = np.exp(-0.5 * ((x - 0.35*n) / (0.05*n))**2)
        template /= np.max(np.abs(template))
        f = np.linspace(0.0, 1.0, n//2 + 1, dtype=np.float64)
        noise_psd = np.abs(1.0/(f+0.02) + 0.5 + 0.02*rng.normal(size=f.size)) + 1e-8
        noise_psd[0] = noise_psd[1]
        src = 'synthetic fallback'
    return template, noise_psd, fs, src


baseline_mod = load_module(BASELINE_PATH, 'of_baseline_module')
clean_mod = load_module(CLEAN_PATH, 'of_clean_module')
template, noise_psd, fs, src = maybe_load_reference_inputs()
of_base = baseline_mod.OptimumFilter(template, noise_psd, fs)
of_clean = clean_mod.OptimumFilter(template, noise_psd, fs)

print('input source:', src)
print('N:', template.size, 'PSD bins:', noise_psd.size, 'fs:', fs)

input source: reference files
N: 32768 PSD bins: 16385 fs: 250000.0


In [3]:
rng = np.random.default_rng(20260312)
N = template.size
allowed = (-40, 40)
n_short = 12
short_traces = []
for _ in range(n_short):
    amp = float(rng.uniform(0.5, 2.0))
    shift = int(rng.integers(allowed[0], allowed[1] + 1))
    tr = amp * np.roll(template, shift) + rng.normal(0.0, 0.03, size=N)
    short_traces.append(tr)
short_traces = np.asarray(short_traces)

L = N + 4096
long_trace = rng.normal(0.0, 0.03, size=L)
for s, a in [(0, 1.0), (1024, 0.8), (2048, 1.3), (3072, 0.7)]:
    if s + N <= L:
        long_trace[s:s+N] += a * template

idx = np.linspace(0, L - N, 12, dtype=int)

print('short traces:', short_traces.shape, 'long trace:', long_trace.shape)

short traces: (12, 32768) long trace: (36864,)


In [4]:
# Warm-up JIT before timing
_ = of_base.fit(short_traces[0])
_ = of_clean.fit(short_traces[0])
_ = of_base.fit_with_shift(short_traces[0], allowed_shift_range=allowed)
_ = of_clean.fit_with_shift(short_traces[0], allowed_shift_range=allowed)
_ = of_base.sliding_fit(long_trace, hop=1, reanchor_every=256, chisq_mode='all')
_ = of_clean.sliding_fit(long_trace, hop=1, reanchor_every=256, chisq_mode='all')
_ = of_base.sliding_fit(long_trace, hop=16, reanchor_every=256, chisq_mode='all')
_ = of_clean.sliding_fit(long_trace, hop=16, reanchor_every=256, chisq_mode='all')
_ = of_base.chisq_at_indices(long_trace, idx)
_ = of_clean.chisq_at_indices(long_trace, idx)
print('warm-up done')

warm-up done


In [5]:
# Numerical equivalence on shared-valid paths
def max_abs(a, b):
    return float(np.max(np.abs(np.asarray(a) - np.asarray(b))))

fit_amp_d = []
fit_chi_d = []
fws_amp_d = []
fws_chi_d = []
fws_t0_d = []
for tr in short_traces:
    a0, c0 = of_base.fit(tr)
    a1, c1 = of_clean.fit(tr)
    fit_amp_d.append(abs(a0-a1))
    fit_chi_d.append(abs(c0-c1))

    a0s, c0s, t0 = of_base.fit_with_shift(tr, allowed_shift_range=allowed)
    a1s, c1s, t1 = of_clean.fit_with_shift(tr, allowed_shift_range=allowed)
    fws_amp_d.append(abs(a0s-a1s))
    fws_chi_d.append(abs(c0s-c1s))
    fws_t0_d.append(abs(t0-t1))

amp_b_h1, chi_b_h1 = of_base.sliding_fit(long_trace, hop=1, reanchor_every=256, chisq_mode='all')
amp_c_h1, chi_c_h1 = of_clean.sliding_fit(long_trace, hop=1, reanchor_every=256, chisq_mode='all')
amp_b_h16, chi_b_h16 = of_base.sliding_fit(long_trace, hop=16, reanchor_every=256, chisq_mode='all')
amp_c_h16, chi_c_h16 = of_clean.sliding_fit(long_trace, hop=16, reanchor_every=256, chisq_mode='all')
chi_b_idx, amp_b_idx = of_base.chisq_at_indices(long_trace, idx)
chi_c_idx, amp_c_idx = of_clean.chisq_at_indices(long_trace, idx)

summary = {
    'fit_amp_maxdiff': max(fit_amp_d),
    'fit_chi_maxdiff': max(fit_chi_d),
    'fit_with_shift_amp_maxdiff': max(fws_amp_d),
    'fit_with_shift_chi_maxdiff': max(fws_chi_d),
    'fit_with_shift_t0_maxdiff': int(max(fws_t0_d)),
    'sliding_hop1_amp_maxdiff': max_abs(amp_b_h1, amp_c_h1),
    'sliding_hop1_chi_maxdiff': max_abs(chi_b_h1, chi_c_h1),
    'sliding_hop16_amp_maxdiff': max_abs(amp_b_h16, amp_c_h16),
    'sliding_hop16_chi_maxdiff': max_abs(chi_b_h16, chi_c_h16),
    'chisq_at_indices_amp_maxdiff': max_abs(amp_b_idx, amp_c_idx),
    'chisq_at_indices_chi_maxdiff': max_abs(chi_b_idx, chi_c_idx),
}
summary

{'fit_amp_maxdiff': 0.0,
 'fit_chi_maxdiff': 0.0,
 'fit_with_shift_amp_maxdiff': 0.0,
 'fit_with_shift_chi_maxdiff': 0.0,
 'fit_with_shift_t0_maxdiff': 0,
 'sliding_hop1_amp_maxdiff': 0.0,
 'sliding_hop1_chi_maxdiff': 0.0,
 'sliding_hop16_amp_maxdiff': 0.0,
 'sliding_hop16_chi_maxdiff': 0.0,
 'chisq_at_indices_amp_maxdiff': 0.0,
 'chisq_at_indices_chi_maxdiff': 0.0}

In [6]:
# Report intentional behavior difference in chisq_mode='none'
none_report = {}
try:
    _abn1, _cbn1 = of_base.sliding_fit(long_trace, hop=1, reanchor_every=256, chisq_mode='none')
    none_report['baseline_hop1_none'] = 'ok'
except Exception as exc:
    none_report['baseline_hop1_none'] = f'error: {type(exc).__name__}: {exc}'

try:
    _acn1, ccn1 = of_clean.sliding_fit(long_trace, hop=1, reanchor_every=256, chisq_mode='none')
    none_report['clean_hop1_none'] = f'ok; all_nan={bool(np.all(np.isnan(ccn1)))}'
except Exception as exc:
    none_report['clean_hop1_none'] = f'error: {type(exc).__name__}: {exc}'

abn16, cbn16 = of_base.sliding_fit(long_trace, hop=16, reanchor_every=256, chisq_mode='none')
acn16, ccn16 = of_clean.sliding_fit(long_trace, hop=16, reanchor_every=256, chisq_mode='none')
none_report['baseline_hop16_none_all_nan'] = bool(np.all(np.isnan(cbn16)))
none_report['clean_hop16_none_all_nan'] = bool(np.all(np.isnan(ccn16)))
none_report['hop16_none_amp_maxdiff'] = float(np.max(np.abs(abn16-acn16)))
none_report

{'baseline_hop1_none': 'error: TypeError: not enough arguments: expected 13, got 12',
 'clean_hop1_none': 'ok; all_nan=True',
 'baseline_hop16_none_all_nan': False,
 'clean_hop16_none_all_nan': True,
 'hop16_none_amp_maxdiff': 0.0}

In [7]:
# Performance benchmark (post warm-up)
def bench(fn, repeats=5):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return {
        'mean_ms': 1000.0 * statistics.mean(times),
        'stdev_ms': 1000.0 * (statistics.stdev(times) if len(times) > 1 else 0.0),
        'min_ms': 1000.0 * min(times),
    }

perf = {}
perf['fit_baseline'] = bench(lambda: [of_base.fit(tr) for tr in short_traces], repeats=8)
perf['fit_clean'] = bench(lambda: [of_clean.fit(tr) for tr in short_traces], repeats=8)
perf['fit_with_shift_baseline'] = bench(lambda: [of_base.fit_with_shift(tr, allowed_shift_range=allowed) for tr in short_traces], repeats=6)
perf['fit_with_shift_clean'] = bench(lambda: [of_clean.fit_with_shift(tr, allowed_shift_range=allowed) for tr in short_traces], repeats=6)
perf['sliding_hop1_all_baseline'] = bench(lambda: of_base.sliding_fit(long_trace, hop=1, reanchor_every=256, chisq_mode='all'), repeats=4)
perf['sliding_hop1_all_clean'] = bench(lambda: of_clean.sliding_fit(long_trace, hop=1, reanchor_every=256, chisq_mode='all'), repeats=4)
perf['sliding_hop16_all_baseline'] = bench(lambda: of_base.sliding_fit(long_trace, hop=16, reanchor_every=256, chisq_mode='all'), repeats=6)
perf['sliding_hop16_all_clean'] = bench(lambda: of_clean.sliding_fit(long_trace, hop=16, reanchor_every=256, chisq_mode='all'), repeats=6)
perf['chisq_at_indices_baseline'] = bench(lambda: of_base.chisq_at_indices(long_trace, idx), repeats=12)
perf['chisq_at_indices_clean'] = bench(lambda: of_clean.chisq_at_indices(long_trace, idx), repeats=12)

def ratio(a, b):
    return perf[b]['mean_ms'] / perf[a]['mean_ms']

perf_ratios = {
    'fit_clean_vs_baseline': ratio('fit_baseline', 'fit_clean'),
    'fit_with_shift_clean_vs_baseline': ratio('fit_with_shift_baseline', 'fit_with_shift_clean'),
    'sliding_hop1_all_clean_vs_baseline': ratio('sliding_hop1_all_baseline', 'sliding_hop1_all_clean'),
    'sliding_hop16_all_clean_vs_baseline': ratio('sliding_hop16_all_baseline', 'sliding_hop16_all_clean'),
    'chisq_at_indices_clean_vs_baseline': ratio('chisq_at_indices_baseline', 'chisq_at_indices_clean'),
}
perf, perf_ratios

({'fit_baseline': {'mean_ms': 4.958883255312685,
   'stdev_ms': 0.25594583627779205,
   'min_ms': 4.780690011102706},
  'fit_clean': {'mean_ms': 4.692540896940045,
   'stdev_ms': 0.059193669627342925,
   'min_ms': 4.639907041564584},
  'fit_with_shift_baseline': {'mean_ms': 7.04782433846655,
   'stdev_ms': 0.04927974046761896,
   'min_ms': 7.008391956333071},
  'fit_with_shift_clean': {'mean_ms': 7.067724674319227,
   'stdev_ms': 0.01336779005242657,
   'min_ms': 7.053563022054732},
  'sliding_hop1_all_baseline': {'mean_ms': 70.92574948910624,
   'stdev_ms': 0.5615937838900018,
   'min_ms': 70.48392598517239},
  'sliding_hop1_all_clean': {'mean_ms': 74.21411148970947,
   'stdev_ms': 5.479310102760511,
   'min_ms': 71.45164400571957},
  'sliding_hop16_all_baseline': {'mean_ms': 45.51499799708836,
   'stdev_ms': 0.21388464284699918,
   'min_ms': 45.319911965634674},
  'sliding_hop16_all_clean': {'mean_ms': 45.00963865818145,
   'stdev_ms': 0.08731597772042782,
   'min_ms': 44.91513600805

In [8]:
# Smoke assertions (shared-valid paths only)
assert summary['fit_amp_maxdiff'] < 1e-12
assert summary['fit_chi_maxdiff'] < 1e-18
assert summary['fit_with_shift_amp_maxdiff'] < 1e-12
assert summary['fit_with_shift_chi_maxdiff'] < 1e-18
assert summary['fit_with_shift_t0_maxdiff'] == 0
assert summary['sliding_hop1_amp_maxdiff'] < 1e-10
assert summary['sliding_hop1_chi_maxdiff'] < 1e-18
assert summary['sliding_hop16_amp_maxdiff'] < 1e-10
assert summary['sliding_hop16_chi_maxdiff'] < 1e-18
assert summary['chisq_at_indices_amp_maxdiff'] < 1e-12
assert summary['chisq_at_indices_chi_maxdiff'] < 1e-18
print('PASS: cleaned module matches baseline on shared-valid paths.')

PASS: cleaned module matches baseline on shared-valid paths.
